In [1]:
# 전처리된 음성데이터로 학습을 진행합니다.
# 학습된 모델은 '비트메이트_TP01/qwen3_ft_output'에 저장됩니다.
# 학습시에는 자원을 많이 먹습니다. A100 추천

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 설치 + 경로 설정 + 리포/모델 준비
from pathlib import Path
import subprocess
import sys
import os

# 기본 경로
PROJECT_DIR = Path("/content/drive/MyDrive/비트메이트_TP01")
SPEAKER = "jhc100"

DATASET_DIR = PROJECT_DIR / "data" / "dataset" / f"wav_{SPEAKER}"
WAVS_DIR = DATASET_DIR / "wavs"
METADATA_PATH = DATASET_DIR / "metadata.txt"
REF_AUDIO_PATH = DATASET_DIR / "ref.wav"

MODEL_DIR = PROJECT_DIR / "models" / "Qwen3-TTS-12Hz-1.7B-Base"
TRAIN_RAW_JSONL = PROJECT_DIR / "train_raw.jsonl"
TRAIN_CODES_JSONL = PROJECT_DIR / "train_with_codes.jsonl"
LOG_DIR = PROJECT_DIR / "qwen3_logs"
FT_OUTPUT_DIR = PROJECT_DIR / "qwen3_ft_output" / SPEAKER

REPO_DIR = Path("/content/Qwen3-TTS")
FINETUNE_DIR = REPO_DIR / "finetuning"

# 설치 함수
def run(cmd, cwd=None):
    print(">", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True, cwd=str(cwd) if cwd else None)

run([sys.executable, "-m", "pip", "install", "-U", "qwen-tts", "huggingface_hub"])
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "sox", "libsox-fmt-all"])

# Qwen3-TTS repo clone
if not REPO_DIR.exists():
    run(["git", "clone", "https://github.com/QwenLM/Qwen3-TTS.git", str(REPO_DIR)])

# base model 다운로드
from huggingface_hub import snapshot_download

MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not any(MODEL_DIR.iterdir()):
    snapshot_download(
        repo_id="Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        local_dir=str(MODEL_DIR),
        local_dir_use_symlinks=False,
    )

print("PROJECT_DIR :", PROJECT_DIR)
print("DATASET_DIR :", DATASET_DIR)
print("MODEL_DIR   :", MODEL_DIR)
print("FINETUNE_DIR:", FINETUNE_DIR)

> /usr/bin/python3 -m pip install -U qwen-tts huggingface_hub
> apt-get update -qq
> apt-get install -y sox libsox-fmt-all
> git clone https://github.com/QwenLM/Qwen3-TTS.git /content/Qwen3-TTS


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

speech_tokenizer/model.safetensors:   0%|          | 0.00/682M [00:00<?, ?B/s]

PROJECT_DIR : /content/drive/MyDrive/비트메이트_TP01
DATASET_DIR : /content/drive/MyDrive/비트메이트_TP01/data/dataset/wav_jhc100
MODEL_DIR   : /content/drive/MyDrive/비트메이트_TP01/models/Qwen3-TTS-12Hz-1.7B-Base
FINETUNE_DIR: /content/Qwen3-TTS/finetuning


In [4]:
# metadata.txt → train_raw.jsonl 변환
from pathlib import Path
import json

# 필수 파일 체크
for p in [METADATA_PATH, REF_AUDIO_PATH, WAVS_DIR]:
    if not p.exists():
        raise FileNotFoundError(f"필수 경로 없음: {p}")

rows = []
missing_files = []

with METADATA_PATH.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        if "|" not in line:
            print(f"[스킵] {line_no}번째 줄 형식 이상: {line}")
            continue

        fname, text = line.split("|", 1)
        audio_path = WAVS_DIR / fname

        if not audio_path.exists():
            missing_files.append(audio_path)
            continue

        rows.append({
            "audio": str(audio_path),
            "text": text.strip(),
            "ref_audio": str(REF_AUDIO_PATH),
        })

with TRAIN_RAW_JSONL.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("저장 완료:", TRAIN_RAW_JSONL)
print("유효 샘플 수:", len(rows))

if missing_files:
    print("\n없는 파일들(최대 20개):")
    for p in missing_files[:20]:
        print("-", p)

저장 완료: /content/drive/MyDrive/비트메이트_TP01/train_raw.jsonl
유효 샘플 수: 100


In [6]:
# 데이터 준비 + sft 스크립트 패치 + 학습
from pathlib import Path

# 1) prepare_data 실행
run([
    sys.executable, "prepare_data.py",
    "--device", "cuda:0",
    "--tokenizer_model_path", "Qwen/Qwen3-TTS-Tokenizer-12Hz",
    "--input_jsonl", TRAIN_RAW_JSONL,
    "--output_jsonl", TRAIN_CODES_JSONL,
], cwd=FINETUNE_DIR)

# 2) Colab용 패치
SFT_SCRIPT = FINETUNE_DIR / "sft_12hz.py"
text = SFT_SCRIPT.read_text(encoding="utf-8")

text = text.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
text = text.replace("attn_implementation='flash_attention_2'", "attn_implementation='sdpa'")

old = 'accelerator = Accelerator(gradient_accumulation_steps=4, mixed_precision="bf16", log_with="tensorboard")'
new = '''accelerator = Accelerator(
        gradient_accumulation_steps=4,
        mixed_precision="bf16",
        log_with="tensorboard",
        project_dir=r"''' + str(LOG_DIR) + '''"
    )'''

if old in text:
    text = text.replace(old, new)

SFT_SCRIPT.write_text(text, encoding="utf-8")
print("sft_12hz.py 패치 완료")

# 3) 학습 파라미터
batch_size = 1
lr = 2e-6
epochs = 4

FT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# 4) 학습 실행
run([
    sys.executable, "sft_12hz.py",
    "--init_model_path", MODEL_DIR,
    "--output_model_path", FT_OUTPUT_DIR,
    "--train_jsonl", TRAIN_CODES_JSONL,
    "--batch_size", str(batch_size),
    "--lr", str(lr),
    "--num_epochs", str(epochs),
    "--speaker_name", SPEAKER,
], cwd=FINETUNE_DIR)

print("학습 완료")
print("출력 폴더:", FT_OUTPUT_DIR)
print("내용:", [p.name for p in FT_OUTPUT_DIR.iterdir()])

> /usr/bin/python3 prepare_data.py --device cuda:0 --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz --input_jsonl /content/drive/MyDrive/비트메이트_TP01/train_raw.jsonl --output_jsonl /content/drive/MyDrive/비트메이트_TP01/train_with_codes.jsonl
sft_12hz.py 패치 완료
> /usr/bin/python3 sft_12hz.py --init_model_path /content/drive/MyDrive/비트메이트_TP01/models/Qwen3-TTS-12Hz-1.7B-Base --output_model_path /content/drive/MyDrive/비트메이트_TP01/qwen3_ft_output/jhc100 --train_jsonl /content/drive/MyDrive/비트메이트_TP01/train_with_codes.jsonl --batch_size 1 --lr 2e-06 --num_epochs 4 --speaker_name jhc100
학습 완료
출력 폴더: /content/drive/MyDrive/비트메이트_TP01/qwen3_ft_output/jhc100
내용: ['checkpoint-epoch-0', 'checkpoint-epoch-1', 'checkpoint-epoch-2', 'checkpoint-epoch-3']


In [7]:
# 체크포인트 테스트
from pathlib import Path
import torch
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

TEST_TEXT = (
    "토끼와 거북이는 빠른 토끼가 느린 거북이를 얕보고 경주 도중 방심해 잠이 드는 사이, "
    "꾸준히 쉬지 않고 나아간 거북이가 결국 먼저 도착해 승리한다는 이야기로, "
    "재능이나 속도보다도 끝까지 성실하게 노력하는 태도가 더 중요하다는 교훈을 전한다."
)

checkpoint_dirs = sorted([
    p for p in FT_OUTPUT_DIR.iterdir()
    if p.is_dir() and p.name.startswith("checkpoint-epoch-")
])

if not checkpoint_dirs:
    raise FileNotFoundError(f"체크포인트 폴더가 없습니다: {FT_OUTPUT_DIR}")

for ckpt_path in checkpoint_dirs:
    weight_file = ckpt_path / "model.safetensors"
    if not weight_file.exists():
        print(f"{ckpt_path.name} -> skip (no weights)")
        continue

    print(f"\n===== {ckpt_path.name} =====")

    tts = Qwen3TTSModel.from_pretrained(
        str(ckpt_path),
        device_map="cuda:0" if torch.cuda.is_available() else "cpu",
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )

    wavs, sr = tts.generate_custom_voice(
        text=TEST_TEXT,
        speaker=SPEAKER,
    )

    out_path = FT_OUTPUT_DIR / f"{ckpt_path.name}.wav"
    sf.write(str(out_path), wavs[0], sr)

    print("저장:", out_path)
    display(Audio(str(out_path)))


********
********
 
checkpoint-epoch-0 -> skip (no weights)

===== checkpoint-epoch-1 =====


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


저장: /content/drive/MyDrive/비트메이트_TP01/qwen3_ft_output/jhc100/checkpoint-epoch-1.wav



===== checkpoint-epoch-2 =====


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


저장: /content/drive/MyDrive/비트메이트_TP01/qwen3_ft_output/jhc100/checkpoint-epoch-2.wav



===== checkpoint-epoch-3 =====


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


저장: /content/drive/MyDrive/비트메이트_TP01/qwen3_ft_output/jhc100/checkpoint-epoch-3.wav
